In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("vehicles_dataset.csv")
df.head()

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X = df.drop(columns=['prezzo', 'price'], errors='ignore') # Assicurati del nome corretto della colonna target
y = df['price'] # o 'prezzo'

# Divisione
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 1. RIMOZIONE VALORI NULLI ---
colonne_da_droppare = ['fuel', 'year', 'model', 'transmission', 'title_status', 'odometer', 'manufacturer']

X_train = X_train.dropna(subset=colonne_da_droppare)
y_train = y_train.loc[X_train.index] # Allineiamo le etichette (y) agli indici rimasti in X

X_test = X_test.dropna(subset=colonne_da_droppare)
y_test = y_test.loc[X_test.index]

# --- 2. RIEMPIMENTO VALORI NULLI CON 'Unknown' (Corretto) ---
# Attenzione: lo facciamo su X_train e X_test, non su df!
colonne_da_riempire = ['condition', 'cylinders', 'drive', 'size', 'type', 'paint_color']

X_train[colonne_da_riempire] = X_train[colonne_da_riempire].fillna('Unknown')
X_test[colonne_da_riempire] = X_test[colonne_da_riempire].fillna('Unknown')

print(f"Dimensioni finali X_train: {X_train.shape}")
print(f"Dimensioni finali X_test: {X_test.shape}")

# --- 3. SALVATAGGIO IN CSV ---
print("Salvataggio dei file CSV in corso...")

# index=False evita di salvare la numerazione delle righe come colonna aggiuntiva nel CSV
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

# y_train e y_test sono Series, ma il metodo to_csv funziona allo stesso modo
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Tutti e 4 i dataset sono stati salvati con successo!")

Dimensioni finali X_train: (311786, 14)
Dimensioni finali X_test: (77818, 14)
Salvataggio dei file CSV in corso...
Tutti e 4 i dataset sono stati salvati con successo!


# Random Forest 

IL Random Forest non accetta valori stringhe o categorie, quindi vanno trasformati tutti i valori prima di poter applicare il modello.

In [2]:
# trasformazione colonna per colonna 
# in ordine year, facciamo la differenza tra anno corrente e anno dell' auto e sovrascriviamo la colonna 
# In questo modo il numero parte da 0 a salire per valori interi e il modello dovrebbe capire che man mano che sale 

from datetime import datetime

# Otteniamo l'anno corrente in automatico
anno_corrente = datetime.now().year
print(anno_corrente)

# Creiamo la nuova feature "eta_auto" sottraendo l'anno di immatricolazione
X_train['eta_auto'] = (anno_corrente - X_train['year']).astype(int)
X_test['eta_auto'] = (anno_corrente - X_test['year']).astype(int)

# (Opzionale) Una volta calcolata l'età, l'anno originale potrebbe essere ridondante 
# per i modelli, quindi puoi decidere di rimuovere la colonna originale
X_train = X_train.drop(columns=['year'])
X_test = X_test.drop(columns=['year'])

print(X_train.head())

2026
       manufacturer                  model  condition    cylinders fuel  \
366318         ford                  f-150    Unknown  8 cylinders  gas   
56271         acura                    mdx    Unknown  6 cylinders  gas   
264620         ford       f-250 super duty  excellent  8 cylinders  gas   
341412        buick  enclave essence sport       good  6 cylinders  gas   
153889         ford        f150 fx/2 sport  excellent  8 cylinders  gas   

        odometer title_status transmission drive       size    type  \
366318   48544.0        clean    automatic   4wd    Unknown   truck   
56271    53103.0        clean    automatic   fwd    Unknown     SUV   
264620   73394.0        clean    automatic   4wd  full-size   truck   
341412   38459.0        clean        other   fwd    Unknown     SUV   
153889  173267.0        clean    automatic   rwd  full-size  pickup   

       paint_color state  eta_auto  
366318       black    tx         9  
56271      Unknown    ca         8  
264620

In [ ]:
# Passiamo ai cilindri sono facili da convertire mettiamo solo i numeri dei cilindri, visualizziamo quali sono i valori unici. 
X_train['cylinders'].unique()